In [2]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense,Layer, Dropout,Conv2D, MaxPooling2D, Flatten
from tensorflow.keras.datasets import mnist 
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.applications import vgg16, vgg19, inception_v3, resnet50
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator


In [2]:
(x_train, y_train), (x_test, y_test) = mnist.load_data()
x_train.shape

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


(60000, 28, 28)

In [3]:
(x_train.shape[1], x_train.shape[2]) 

(28, 28)

In [4]:
model = Sequential([
    Conv2D(filters=32 ,kernel_size=(3,3), activation="relu",input_shape= (28,28,1)),
    Conv2D(filters=32, kernel_size=(3,3), activation="relu", padding="same", strides=(2,2)),
    MaxPooling2D(pool_size=(3,3)),
    Flatten(),

    Dense(64, activation="relu"),
    Dense(32, activation="relu"),
    Dense(16, activation="softmax")
])
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 26, 26, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 13, 13, 32)     │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 4, 4, 32)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │        32,832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 16)             │           528 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 45,008 (175.81 KB)

 Trainable params: 45,008 (175.81 KB)

 Non-trainable params: 0 (0.00 B)

In [5]:
model.compile(
    optimizer = "adam",
    loss = "sparse_categorical_crossentropy",
    metrics = ["accuracy"]
)
Early_Stopping = EarlyStopping(patience=2, monitor="val_loss")
model.fit(x_train, y_train,
          epochs = 2,
          validation_split = 0.1,
          callbacks = Early_Stopping
          )

Epoch 1/2
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 40s 23ms/step - accuracy: 0.9134 - loss: 0.3216 - val_accuracy: 0.9712 - val_loss: 0.0916
Epoch 2/2
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 40s 22ms/step - accuracy: 0.9727 - loss: 0.0879 - val_accuracy: 0.9822 - val_loss: 0.0599


In [6]:
model.evaluate(x_test, y_test)

313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.9779 - loss: 0.0663


[0.06626398861408234, 0.9779000282287598]

In [3]:
import kagglehub
train_path = kagglehub.dataset_download("abdullahkhanuet22/eggs-images-classification-damaged-or-not")

Using Colab cache for faster access to the 'eggs-images-classification-damaged-or-not' dataset.


In [6]:
import os

print(os.listdir(train_path))

['Eggs Classification']


In [ ]:
train_gen = ImageDataGenerator(   # Data Augmentation
    validation_split = 0.2,   # قسمنا الداته 20% تيست والباقي ترين في حالة الداته لو مش متقسمه
    rescale = 1./255,  # حول البكسلات من 0-255 الي 0-1 علشان الموديل يتعلم اسرع
    rotation_range= 40,  # لف الصوره عشوائي علشان الموديل يتعود ع اختلاف الاتجاه
    horizontal_flip = True, # اقلب الصوره يمين وشمال
    height_shift_range = 0.1, # حرك الصوره لفوق وتحت بنسبة كذا
    width_shift_range = 0.1, # حرك الصوره يمين وشمال 
    brightness_range = (0.5, 1.5), # بقولوا غير اضاءة الصوره علشان يتعود ع اضاءة المكان
    zoom_range = (1, 1.5) # اعملي زوووم عشان يجيب تفاصيل اكتر


)
train_batch = train_gen.flow_from_directory( # اقرا الصور واعمل ليبول تلقائي من اسم الفولدر قسمها لباتش
    directory = train_path, target_size = (224,224),batch_size = 10, subset = "training" #هات الصور من فولدر الداتا الأساسي , صغر\كبر الصور لنفس الحجم, هات 10 صور في كل مره تدريب , خد 80% من الدلاه اللي اتقسمت فوق
)

val_batch = train_gen.flow_from_directory(
    directory = train_path, target_size = (224,224),batch_size = 10, subset = "validation"
)

Found 636 images belonging to 1 classes.
Found 158 images belonging to 1 classes.


In [10]:
import os
print(os.path.exists(train_path))

True


In [11]:
print(train_batch.image_shape)

(224, 224, 3)


In [12]:
print("Train images:", train_batch.samples)
print("Validation images:", val_batch.samples)

Train images: 636
Validation images: 158


In [13]:
train_batch[0]

(array([[[[0.37647063, 0.37647063, 0.36862746],
          [0.3647059 , 0.3647059 , 0.3647059 ],
          [0.35686275, 0.36078432, 0.3647059 ],
          ...,
          [0.40000004, 0.40000004, 0.38823533],
          [0.3803922 , 0.38431376, 0.36862746],
          [0.36862746, 0.37254903, 0.35686275]],
 
         [[0.3803922 , 0.3803922 , 0.37647063],
          [0.37647063, 0.37647063, 0.37254903],
          [0.37647063, 0.37647063, 0.36862746],
          ...,
          [0.3921569 , 0.39607847, 0.3803922 ],
          [0.37254903, 0.37647063, 0.36078432],
          [0.36862746, 0.37254903, 0.35686275]],
 
         [[0.37647063, 0.37647063, 0.36862746],
          [0.3803922 , 0.3803922 , 0.37647063],
          [0.3803922 , 0.3803922 , 0.37647063],
          ...,
          [0.38431376, 0.38823533, 0.37254903],
          [0.36862746, 0.37254903, 0.35686275],
          [0.36862746, 0.37254903, 0.35686275]],
 
         ...,
 
         [[0.32941177, 0.34509805, 0.3254902 ],
          [0.33725

In [14]:
model = Sequential([
    Conv2D(filters=32, activation="relu", kernel_size=(3,3),input_shape = (224,224,3)),
    Conv2D(filters=32, activation="relu", kernel_size=(3,3), padding="same", strides=1),
    MaxPooling2D(pool_size=(2,2)),

    Flatten(),
    Dense(32, activation="relu"),
    Dropout(0.3),
    Dense(16, activation="relu"),
    Dropout(0.2),
    Dense(1, activation="sigmoid")
])
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 222, 222, 32)   │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 394272)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │    12,616,736 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 12,627,425 (48.17 MB)

 Trainable params: 12,627,425 (48.17 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
model.compile(
    optimizer = "adam",
    loss = "binary_crossentropy",
    metrics = ["accuracy"]
)
early_stopping = EarlyStopping(monitor= "val_accuracy", patience= 3)
model.fit(train_batch,
          validation_data = val_batch, # evaluate
          epochs = 5,
          callbacks = early_stopping
          )

Epoch 1/5
64/64 ━━━━━━━━━━━━━━━━━━━━ 114s 2s/step - accuracy: 0.9638 - loss: 0.1281 - val_accuracy: 1.0000 - val_loss: 3.1534e-07
Epoch 2/5
64/64 ━━━━━━━━━━━━━━━━━━━━ 143s 2s/step - accuracy: 0.9937 - loss: 0.0197 - val_accuracy: 1.0000 - val_loss: 4.7902e-17
Epoch 3/5
64/64 ━━━━━━━━━━━━━━━━━━━━ 113s 2s/step - accuracy: 0.9921 - loss: 0.0331 - val_accuracy: 1.0000 - val_loss: 4.4256e-29
Epoch 4/5
64/64 ━━━━━━━━━━━━━━━━━━━━ 141s 2s/step - accuracy: 1.0000 - loss: 0.0020 - val_accuracy: 1.0000 - val_loss: 4.0391e-27


In [16]:
model.evaluate(val_batch)

16/16 ━━━━━━━━━━━━━━━━━━━━ 11s 669ms/step - accuracy: 1.0000 - loss: 3.8111e-38


[3.8111377778353343e-38, 1.0]